# Xarray with browser-backed Icechunk I/O

`ipygis` is a bridge to GIS libraries running in the browser. There are a couple of things to know to have it working:

- you must use the `xeus-python` kernel (`ipykernel` currently has a limitation with top-level await and widgets).
- the remote server, or a range-preserving proxy, must allow cross-origin browser requests.
- when using the `@earthmover/icechunk` WASM library, the server must send COOP/COEP headers so that `SharedArrayBuffer` is supported in the browser.

In [ ]:
import xarray as xr
from ipygis.icechunk import Repository, jupyter_storage
from ipygis.zarr.codecs import BrowserLzwCodec  # registers imagecodecs_lzw
import zarr

zarr.config.set({"codecs.imagecodecs_lzw": "ipygis.zarr.codecs.BrowserLzwCodec"});  # choose our codec

In [ ]:
storage = jupyter_storage("examples/hydrosheds.icechunk")
repository = await Repository.open_async(
    storage,
    # backend="@earthmover/icechunk",  # "icechunk-js" is the default
    proxy_url="https://my-proxy.david-brochart.workers.dev/",
    virtual_chunk_prefixes=["https://data.hydrosheds.org/file/hydrosheds-v2/ACC/1s/"],
)
session = await repository.readonly_session_async("main")
session.snapshot_id

In [ ]:
ds = xr.open_zarr(
    session.store,
    chunks=None,
    consolidated=False,
    create_default_indexes=False,
)
ds

In [ ]:
selection = ds["0"].isel(tile=10, y=100, x=200)
result = await selection.load_async()
result

In [ ]:
result.item()  # Expected: 7.0

In [ ]:
# Close after finishing all reads from this dataset.
await session.aclose()
await repository.aclose()